# Parser Single Web


In [3]:
import requests

url = "https://medium.com/gitconnected/10-ways-to-write-better-python-codes-55fc862ab0ef?format=json"
response = requests.get(url)
if response.status_code == 200:
    json_data = response.text[16:]  # 去除防止 JSON 劫持的前缀
    print(json_data)
else:
    print(f"Failed to retrieve data: {response.status_code}")

{"success":true,"payload":{"value":{"id":"55fc862ab0ef","versionId":"3bd7223e4517","creatorId":"b1efbf95e646","homeCollectionId":"5517fd7b58a6","title":"10 Ways to Write Better Python Codes","detectedLanguage":"en","latestVersion":"3bd7223e4517","latestPublishedVersion":"3bd7223e4517","hasUnpublishedEdits":false,"latestRev":3416,"createdAt":1721284836996,"updatedAt":1732053170484,"acceptedAt":0,"firstPublishedAt":1722009635724,"latestPublishedAt":1730132230662,"vote":false,"experimentalCss":"","displayAuthor":"","content":{"subtitle":"Everyday Tips and Tricks You Need to Know!","bodyModel":{"paragraphs":[{"name":"6754","type":3,"text":"10 Ways to Write Better Python Codes","markups":[]},{"name":"2433","type":13,"text":"Everyday Tips and Tricks You Need to Know!","markups":[]},{"name":"2108","type":4,"text":"(Image by Author)","markups":[],"layout":1,"metadata":{"id":"1*qEJ1r4n4t-kzjWOjSNhuRQ.png","originalWidth":1168,"originalHeight":682,"alt":"Python Block — (Image by Author)"}},{"nam

In [4]:
import json

data = json.loads(json_data)
data

{'success': True,
 'payload': {'value': {'id': '55fc862ab0ef',
   'versionId': '3bd7223e4517',
   'creatorId': 'b1efbf95e646',
   'homeCollectionId': '5517fd7b58a6',
   'title': '10 Ways to Write Better Python Codes',
   'detectedLanguage': 'en',
   'latestVersion': '3bd7223e4517',
   'latestPublishedVersion': '3bd7223e4517',
   'hasUnpublishedEdits': False,
   'latestRev': 3416,
   'createdAt': 1721284836996,
   'updatedAt': 1732053170484,
   'acceptedAt': 0,
   'firstPublishedAt': 1722009635724,
   'latestPublishedAt': 1730132230662,
   'vote': False,
   'experimentalCss': '',
   'displayAuthor': '',
   'content': {'subtitle': 'Everyday Tips and Tricks You Need to Know!',
    'bodyModel': {'paragraphs': [{'name': '6754',
       'type': 3,
       'text': '10 Ways to Write Better Python Codes',
       'markups': []},
      {'name': '2433',
       'type': 13,
       'text': 'Everyday Tips and Tricks You Need to Know!',
       'markups': []},
      {'name': '2108',
       'type': 4,
    

In [5]:
title = data["payload"]["value"]["title"]
title

'10 Ways to Write Better Python Codes'

In [6]:
publish_date = data["payload"]["value"]["latestPublishedAt"]
publish_date

1730132230662

In [7]:
# 文章標籤
tags_list = []
for key, value in data["payload"]["value"]["virtuals"].items():
    if key == "tags":
        for tag in value:
            tags_list.append(tag["name"])
tags_list

['Python', 'Technology', 'Coding Interviews', 'Data', 'Software Engineering']

In [8]:
# 取本文副標題
sub_titles = data["payload"]["value"]["content"]["subtitle"]
sub_titles

'Everyday Tips and Tricks You Need to Know!'

In [9]:
# 取本文內容
paragraphs = data["payload"]["value"]["content"]["bodyModel"]["paragraphs"]

article_content = ""
for paragraph in paragraphs:
    article_content += paragraph["text"]

print(article_content)

10 Ways to Write Better Python CodesEveryday Tips and Tricks You Need to Know!(Image by Author)We’re almost halfway into 2024, the tech industry is moving faster than ever, due to the rise of Generative AI and Large Language Models.Bonus: Read about the trending Technology behind Generative AI!, here.👈🏻As a data enthusiast and python developer you should be knowing these cool tips and trick to stay relevant in todays competitive market, where lakhs of tech-people were laid off.First thing to note here is that,Why Python?Python is an extremely powerful general purpose programming language, highly suitable for Data Science related tasks — utilizing it’s extensive libraries and awesome frameworks like SciKit-Learn, TensorFlow, PyTorch, etc., due to it’s easy-to-understand syntax.This means having Python under your belt can provide you with flexibility to create large variety of applications, for example, automation tasks or building chatbots to interact with open-source LLMs, especially i

In [78]:
import json


def parse_json_data(json_data):

    data = json.loads(json_data)

    sub_titles = data["payload"]["value"]["content"].get("subtitle", "No Subtitle")
    paragraphs = data["payload"]["value"]["content"]["bodyModel"]["paragraphs"]
    # article_content = "".join(paragraph["text"] for paragraph in paragraphs)

    return {
        "sub_titles": sub_titles,
        # "content": article_content,
    }

In [79]:
import requests


def get_article_content(url):

    url = f"{url}?format=json"

    response = requests.get(url)
    if response.status_code == 200:
        if response.text.startswith("])}"):
            json_data = response.text[16:]
            result = parse_json_data(json_data)
    else:
        raise Exception(f"Failed to retrieve data: {response.status_code}")

    return result


# url = "https://medium.com/p/5eece18c0350"
# url = "https://cryptoairdropped193.medium.com/okx-memefi-airdrop-checker-check-your-rewards-today-5eece18c0350"
# get_article_content(url)

# 爬 24 小時內的文章


In [ ]:
import feedparser
from datetime import datetime, timedelta
from typing import Set

base_url = "https://medium.com/feed/tag/"
existed_article = set()
existed_tags = set()

def parse_24hr_medium_feed(categories: list[str]):
    now = datetime.now()
    yesterday = now - timedelta(days=1)

    parsed_tags: Set[str] = set()
    parse_medium_result = {}

    for category in categories:
        print(f"Processing category: {category}")
        result = []
        feed = feedparser.parse(f"{base_url+category}")

        for entry in feed.entries:
            published_time = datetime(*entry.published_parsed[:6])

            if published_time > yesterday:

                if entry.id in existed_article:
                    print(f"Article `{entry.title}` already existed")
                    continue

                print(f"Processing article: {entry.title}")
                try:
                    parse_result = get_article_content(entry.id)
                    tags = [tag.term for tag in entry.tags]
                    parse_result.update(
                        {
                            "title": entry.title,
                            "link": entry.id,
                            "tags": tags,
                            "published_time": published_time,
                        }
                    )
                    result.append(parse_result)
                    existed_article.add(entry.id)
                    parsed_tags = parsed_tags.union(tags)

                except Exception as e:
                    print(f"Failed to parse article: {e}")
                    parse_result = "failed"

        parse_medium_result[category] = result
        print(f"category: {category}, count: {len(parse_medium_result[category])}")

    return parse_medium_result, parsed_tags


categories = ["technology", "self-improvement", "software-development", "deep-learning", "python"]
categories = ["technology"]


gloabl_medium_result = []
parse_medium_result, parsed_tags = parse_24hr_medium_feed(categories)
print(f"parse_medium_result: {parse_medium_result}, parsed_tags: {parsed_tags}")

gloabl_medium_result.append(parse_medium_result)
print(f"gloabl_medium_result: {gloabl_medium_result}")


# # 第一個終止條件 - 本輪新的 tags 都已經存在於 existed_tags 中
# if set(parsed_tags).issubset(existed_tags):
#     pass
# else:
#     existed_tags.union(parsed_tags)
#     new_tags = [tag for tag in parsed_tags if tag not in existed_tags]  # 新的 tags 用於下一輪的爬蟲

# parse_medium_result, parsed_tags = parse_24hr_medium_feed(new_tags)

Processing category: technology
Article `Instantly Get Free FTX Token $FTT Tokens with This Airdrop Guide!` already existed
Article `How to Get Free OLAS Tokens Through Autonolas $OLAS Airdrop Programs` already existed
Article `How to Claim SIDUS $SIDUS Airdrops on DappRadar: A Quick Guide` already existed
Article `Free XYM Giveaway – Grab Tokens in the Latest Airdrop` already existed
Article `Limited-Time Airdrop: Free Velodrome Finance Tokens for Early Users` already existed
Article `Claim Tether Gold $XAUt Airdrop Fast and Get Free Tokens Instantly` already existed
Article `Get Ready for the 2024 Free ELYSIA $EL Airdrop!` already existed
Article `How to Use DappRadar for Seamless Cobak Token $CBK Airdrop Claims` already existed
Article `How to Get Free Flare $FLR Tokens with No Fees Required` already existed
Article `How to Get KYVE Tokens Free with KYVE Network $KYVE Airdrop and Staking` already existed
category: technology, count: 0
parse_medium_result: {'technology': []}, parsed_

In [85]:
existed_article = set()
existed_tags: Set[str] = set()

max_times = 3

gloabl_medium_result = []
categories = ["technology"]
existed_tags = existed_tags.union(categories)
print(f"existed_tags: {existed_tags}")

for i in range(max_times):
    print(f"\n----------------Processing round: {i+1}----------------\n")
    print(f"    The number of categories: {len(categories)}")

    parse_medium_result, parsed_tags = parse_24hr_medium_feed(categories)
    print(f"    parse_medium_result: {parse_medium_result}, parsed_tags: {parsed_tags}")

    if all(value == [] for value in parse_medium_result.values()):  # 第一個終止條件 - 這一輪查詢沒有出現任何新的文章
        print("parse_medium_result is empty, 結束爬蟲")
        break
    else:
        gloabl_medium_result.append(parse_medium_result)

    print(f"    existed_tags: {existed_tags}")
    print(f"    parsed_tags: {parsed_tags}")
    if set(parsed_tags).issubset(existed_tags):  # 第二個終止條件 - 本輪新的 tags 都已經存在於 existed_tags 中
        print("所有 tags 都已经存在于 existed_tags 中, 結束爬蟲")
        break
    else:
        categories = [tag for tag in parsed_tags if tag not in existed_tags]  # 新的 tags 用於下一輪的爬蟲
        existed_tags = existed_tags.union(parsed_tags)
        print(f"    新的 tags: {categories}")

existed_tags: {'technology'}

----------------Processing round: 1----------------

    The number of categories: 1
Processing category: technology
Processing article: Claim Kyber Network Crystal v2 $KNC (KNC) Airdrop Fast and Start Earning Free Tokens
Processing article: Airdrop Reviews: Pepe $PEPE vs Competitors – Which is Best for Free Tokens?
Processing article: Understanding the Ergo Airdrop Process
Processing article: Why the Neiro Airdrop is Essential for Investors
Processing article: Earn Free XSGD $XSGD Tokens Instantly with This Simple Airdrop
Processing article: Polytrade $TRADE Airdrop is Live! Claim Your Free TRADE Tokens Now
Processing article: Engaging with AST: How Airdrops Work
Processing article: Unlock Hidden Crypto Riches – Free Pixels $PIXEL Tokens Await!
Processing article: Quick Guide to Claim Free MultiversX $EGLD Tokens from Airdrop
Processing article: Claiming the HyperGPT Airdrop: Tips and Tricks
category: technology, count: 10
    parse_medium_result: {'techn

In [90]:
gloabl_medium_result

[{'technology': [{'sub_titles': 'Easy Steps for Accessing Kyber Network Crystal v2 Airdrops',
    'title': 'Claim Kyber Network Crystal v2 $KNC (KNC) Airdrop Fast and Start Earning Free Tokens',
    'link': 'https://medium.com/p/4d26ed88f9e3',
    'tags': ['technology',
     'launch',
     'nft',
     'money',
     'kyber-network-crystal-v2'],
    'published_time': datetime.datetime(2024, 12, 13, 7, 3, 34)},
   {'sub_titles': 'In the digital age, where the currency of the future is being minted in the present, airdrops have emerged as a golden opportunity for…',
    'title': 'Airdrop Reviews: Pepe $PEPE vs Competitors – Which is Best for Free Tokens?',
    'link': 'https://medium.com/p/69a2368e2294',
    'tags': ['technology', 'money', 'pepe', 'nft', 'safe'],
    'published_time': datetime.datetime(2024, 12, 13, 7, 3, 33)},
   {'sub_titles': 'Are you ready to embark on a cosmic journey into the world of decentralized finance? Look no further than Ergo crypto, the Ergol currency…',
    

In [98]:
len(gloabl_medium_result)
pages = 0
for result in gloabl_medium_result:
    for key, value in result.items():
        print(f"key: {key}")
        # print(f"value: {value}")
        pages += len(value)

print(f"pages: {pages}")

key: technology
key: hypergpt
key: income
key: money
key: rewards
key: trust
key: pixel
key: members
key: xsgd
key: pepe
key: polytrade
key: mobile
key: launch
key: safe
key: ergo
key: neiro
key: nft
key: airswap
key: early
key: multiversx
key: free
key: kyber-network-crystal-v2
key: acces
key: terra
key: sui
key: landwolf-0x67
key: avail
key: contentos
key: iexec-rlc
key: bonus
key: enjin-coin
key: earn
key: anchored-coins-aeur
key: request
key: linqai
key: artyfact
key: hopr
key: bfg-token
key: dorafactory
key: evan
key: neutron
key: lift-dollar
key: goatseus-maximus
key: digital-imaging
key: hot-doge
key: iotex
key: hyperliquid
key: earn-money-online
key: league-of-kingdoms-arena
key: gem
key: listings
key: oho
key: makerdao
key: adex
key: pendle
key: dogecoin
key: bitcoin
key: zcash
key: coq-inu
key: kyc
key: equilateral-triangle
key: arkham
key: creamfinance
key: innovation
key: aioz-network
key: binance
key: ape-coin
key: axie-infinity
key: algorand
key: passive
key: bellscoin
ke